In [ ]:
!pip install transformers
!pip install sentencepiece
!pip install datasets

## Local Inference on GPU 
Model page: https://huggingface.co/vesteinn/ScandiBERT

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/vesteinn/ScandiBERT)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("kaggle-scandibert")

import torch
from huggingface_hub import hf_hub_download
from transformers import XLMRobertaTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset, concatenate_datasets

model_name = "vesteinn/ScandiBERT" # Considering no-faroese variant for slight tokenization difference
labels = ["Norwegian", "Non-Scandinavian"]
id2label = {i: l for i, l in enumerate(labels)}
label2id = {l: i for i, l in enumerate(labels)}

# Hard-coded because tokenizer.json is broken
vocab_path = hf_hub_download(repo_id=model_name, filename="sentencepiece.bpe.model")
tokenizer = XLMRobertaTokenizer(vocab_file=vocab_path)


model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels = len(labels),
    id2label = id2label,  
    label2id = label2id,
    ignore_mismatched_sizes = True,
    problem_type = "multi_label_classification"
)

def get_opus_sentences(config, lang_code, num_sentences):
    print(f"Downloading {lang_code} from {config}...")
    ds = load_dataset("Helsinki-NLP/opus-100", config, split="train")
    ds = ds.shuffle(seed=42).select(range(num_sentences))
    return [item[lang_code] for item in ds["translation"]]

sample_norwegian_all = get_opus_sentences("en-no", "no", 375) + get_opus_sentences("en-nn", "nn", 125)

nonscandi_configs = [("en-es", "en"), ("en-es", "es"), ("de-en", "de"), ("en-nl", "nl"), ("af-en", "af")]
sample_nonscandi_all = []
for config, code in nonscandi_configs:
    sample_nonscandi_all.extend(get_opus_sentences(config, code, 100))
print(f"Non-Scandinavian corpora collected: {len(sample_nonscandi_all)} sentences.")

text = "Dette er en norsk setning."
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
print("\n")
probs = torch.softmax(outputs.logits, dim=-1)
for label, prob in zip(labels, probs[0]): # True "zero-shot", random classification test
    print(f"{label}: {prob.item():.4f}")

#for i in sample_norwegian_all["text"]:
    #print(i)
#for sent in sample_nonscandi_all[:5]:
    #print("-", sent)

sentencepiece.bpe.model:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/523 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: vesteinn/ScandiBERT
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


README.md: 0.00B [00:00, ?B/s]

... (33 outputs left out)

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/275512 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Non-Scandinavian corpora collected: 500 sentences.


Norwegian: 0.5239
Non-Scandinavian: 0.4761


In [3]:
from datasets import Dataset, concatenate_datasets

dataset_norwegian = Dataset.from_dict({"text": sample_norwegian_all, "labels": [[1.0, 0.0]] * 500})
dataset_nonscandi = Dataset.from_dict({"text": sample_nonscandi_all, "labels": [[0.0, 1.0]] * 500})
train_dataset = concatenate_datasets([dataset_norwegian, dataset_nonscandi]).shuffle(seed=42)

print(train_dataset[:20])

def tokenize_dataset(select_dataset):
    return tokenizer(select_dataset["text"], truncation=True, max_length=512)

train_dataset = train_dataset.map(tokenize_dataset, batched=True)
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

{'text': ['- En die verklarings van die bemanning.', 'Belyn regs', 'Ik wil dat je opstaat en wegloopt.', "'n Grafiese program om skyfgebruik mee te analiseer.", '- Kjære? - Ja.', 'Y, más adelante, dice: Así también la resurrección de los muertos.', 'Dat geluid wat je hoorde waren niet ik en Martin.', '- Jeg har aldri hørt om det.', '- Hvorfor vil ingen tro meg?', '- Ja.', '- El gran Descartes.', '(Stygend)', "I know it didn't turn out that way... but you told him to stand his ground, and you'd back him up.", 'Energie wanneer vol', 'Hva brukes de til? Hvis noen vil ha tilbake minnene sine, må de tegne pentagrammet for å tilkalle demonen.', '...And said to the air-hostess woman,', 'Chapleau', 'Hvor er alle sammen?', 'Ring politiet.', 'Går det bra med deg?'], 'labels': [[0.0, 1.0], [0.0, 1.0], [0.0, 1.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0], [0.0, 1.0], [1.0, 0.0], [1.0, 0.0], [0.0, 1.0], [0.0, 1.0], [0.0, 1.0], [0.0, 1.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 0.0], [1.0, 

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [6]:
import re
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./scandiprob_checkpoints", # Where to save the model weights
    learning_rate=2e-5,                    # 0.00002 learning rate
    per_device_train_batch_size=8,         # Adjust to 4 if Kaggle throws an OutOfMemory (OOM) error
    num_train_epochs=3,                   
    weight_decay=0.01,                     
    report_to="none",
    dataloader_pin_memory=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator
    # eval_dataset=eval_dataset,
)

def nonscandi_penalty(text): # Regex for negative class
    if not text:
        return 0.0

    scandi_keyboard = r"[a-zA-ZæøåÆØÅäöÄÖéÉ0-9 !@#$%^&*()\-_=+\[\]{};':\",.<>?/`~\\|]"
    scandi_matches = re.findall(scandi_keyboard, text)
    nonscandi_percent = (1 - (len(scandi_matches) / len(text)))
    return nonscandi_percent


print("Training...")
trainer.train()
trainer.save_model("./scandiprob_current")
print("Training complete and model saved!")

inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)
print("\n")
raw_probs = torch.sigmoid(outputs.logits)[0]

nonscandi_ratio = nonscandi_penalty(text)


final_probs = []

for i, label in enumerate(labels):
    prob = raw_probs[i].item()
    
    if label == "Norwegian":
        adjusted = prob * (1.0 - nonscandi_ratio) 
    elif label == "Non-Scandinavian": # Scale up/down by % of non-Scandi chars
        adjusted = prob + ((1.0 - prob) * nonscandi_ratio)
        
    final_probs.append(adjusted)

for label, prob in zip(labels, final_probs): # Print results!
    print(f"{label}: {prob:.4f}")

Training...


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete and model saved!


Norwegian: 0.5430
Non-Scandinavian: 0.4601
